# Amazon Entity Resolution — Kaggle training (V7)

This notebook clones the project code, installs its pinned dependencies, and trains LightGBM using **all supplied training entities**. It samples only easy negative candidate pairs, as documented by the pipeline; it does not subset the source tables.

## Before running

1. Import this notebook into Kaggle from the GitHub repository.
2. In the notebook editor, choose **Add Input** and attach the official competition data (accept the competition rules if prompted). Keep the provided `train/` and `test/` folders together.
3. Select a CPU runtime with enough free working-disk space. GPU is not required. The candidate cap is configurable; review the validation candidate recall and oracle score before increasing it.
4. Run cells in order. Outputs are written under `/kaggle/working/v7` and `/kaggle/working/output`.

The training data is read from Kaggle Input rather than copied into GitHub. This lets you use challenge data under Kaggle’s access controls and avoids downloading a multi-gigabyte archive from GitHub on every run.


In [ ]:
from pathlib import Path
import base64, os, subprocess, sys

REPO_URL = 'https://github.com/rohitkumar8279/AMAZON-ML-CHALLENGE.git'
REPO_DIR = Path('/kaggle/working/AMAZON-ML-CHALLENGE')
try:
    from kaggle_secrets import UserSecretsClient
    github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
except Exception as exc:
    raise RuntimeError('Add a Kaggle Secret named GITHUB_TOKEN with read-only access to the private GitHub repository, then rerun this cell.') from exc
auth_env = os.environ.copy()
auth_env['GIT_CONFIG_COUNT'] = '1'
auth_env['GIT_CONFIG_KEY_0'] = 'http.https://github.com/.extraheader'
auth_env['GIT_CONFIG_VALUE_0'] = 'AUTHORIZATION: basic ' + base64.b64encode(('x-access-token:' + github_token).encode()).decode()
auth_env['GIT_LFS_SKIP_SMUDGE'] = '1'  # avoid downloading the 1.09 GB archive unless needed
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True, env=auth_env)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True, env=auth_env)
print('Project:', REPO_DIR)
print('Training script:', REPO_DIR / 'code/business_entity_resolution/src/train_v7.py')


In [ ]:
REQ = REPO_DIR / 'code/business_entity_resolution/requirements.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQ)], check=True)
print('Pinned project dependencies installed.')


In [ ]:
from pathlib import Path
import subprocess, zipfile

INPUT_ROOT = Path('/kaggle/input')
TRAIN_REQUIRED = {'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv'}
TEST_REQUIRED = {'test_source1.tsv', 'test_source2.tsv', 'test_source3.tsv'}
def find_attached_dataset():
    found = []
    if INPUT_ROOT.exists():
        for source1 in INPUT_ROOT.rglob('train_source1.tsv'):
            train_dir = source1.parent
            test_dir = train_dir.parent / 'test'
            if TRAIN_REQUIRED.issubset({p.name for p in train_dir.iterdir() if p.is_file()}) and test_dir.is_dir() and TEST_REQUIRED.issubset({p.name for p in test_dir.iterdir() if p.is_file()}):
                found.append((train_dir, test_dir))
    return found
matches = find_attached_dataset()
if not matches:
    archive = REPO_DIR / '6ab10eb3b23ba_student_resource.zip'
    download_env = auth_env.copy()
    download_env.pop('GIT_LFS_SKIP_SMUDGE', None)
    subprocess.run(['git', '-C', str(REPO_DIR), 'lfs', 'pull', '--include=6ab10eb3b23ba_student_resource.zip'], check=True, env=download_env)
    if not archive.is_file() or archive.stat().st_size < 1_000_000_000:
        raise RuntimeError('Git LFS archive was not downloaded. Ensure Git LFS is available and GITHUB_TOKEN has repository Contents: read access.')
    extract_root = Path('/kaggle/working/challenge_data')
    extract_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(extract_root)
    data_root = extract_root / 'student_resource' / 'dataset'
    matches = [(data_root / 'train', data_root / 'test')]
if len(matches) != 1:
    raise RuntimeError(f'Expected one complete challenge dataset; found {len(matches)}. Attach it through Add Input or verify the private GitHub LFS archive.')
TRAIN_DIR, TEST_DIR = matches[0]
print('Train:', TRAIN_DIR)
print('Test: ', TEST_DIR)
for p in sorted(TRAIN_DIR.iterdir()):
    if p.is_file(): print(f'{p.name}: {p.stat().st_size / 1e9:.2f} GB')


## Train and select thresholds

The default candidate cap is 64 per target source. This bounds expensive pair features while preserving the full entity tables. The saved validation report includes candidate recall and the candidate-only oracle F0.5; those values diagnose blocking losses separately from model errors.

In [ ]:
import os
WORK_DIR = Path('/kaggle/working/v7')
WORK_DIR.mkdir(parents=True, exist_ok=True)
MAX_CANDIDATES_PER_SOURCE = 64
HARD_NEGATIVES_PER_SOURCE = 8
cmd = [sys.executable, str(REPO_DIR / 'code/business_entity_resolution/src/train_v7.py'),
       '--train-dir', str(TRAIN_DIR), '--work-dir', str(WORK_DIR),
       '--max-candidates-per-source', str(MAX_CANDIDATES_PER_SOURCE),
       '--hard-negatives-per-source', str(HARD_NEGATIVES_PER_SOURCE)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
report_path = WORK_DIR / 'validation_report_v7.json'
print(report_path.read_text() if report_path.exists() else 'Training completed; report not found at expected path.')


## Full test inference and submission files

Run after successful training. This processes the complete test tables using the same candidate cap and saved source thresholds. Then download the two TSVs from the Kaggle notebook Output panel. Validate them with the included challenge validator before submission.

In [ ]:
OUT_DIR = Path('/kaggle/working/output')
OUT_DIR.mkdir(parents=True, exist_ok=True)
cmd = [sys.executable, str(REPO_DIR / 'code/business_entity_resolution/src/predict_v7.py'),
       '--test-dir', str(TEST_DIR), '--model-dir', str(WORK_DIR),
       '--output-dir', str(OUT_DIR)]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)
for name in ('matching_results.tsv', 'candidate_pairs.tsv'):
    path = OUT_DIR / name
    print(name, f'{path.stat().st_size / 1e6:.1f} MB' if path.exists() else 'MISSING')


In [ ]:
validator = REPO_DIR / 'utils/validate_submission.py'
cmd = [sys.executable, str(validator), '--matching', str(OUT_DIR / 'matching_results.tsv'),
       '--candidate', str(OUT_DIR / 'candidate_pairs.tsv'), '--test-dir', str(TEST_DIR)]
subprocess.run(cmd, cwd=REPO_DIR, check=True)
